In [ ]:
# ============================================================
# Cell 1 — Install & Mount

In [1]:
# ============================================================
# No extra installs needed — all standard Colab libraries
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✅")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ============================================================
# Cell 2 — Config

In [ ]:
# ============================================================
import cv2
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

CONFIG = {
    "image_dir":       "/content/drive/MyDrive/bacterial_leaf_blight",  # original images
    "mask_dir":        "/content/drive/MyDrive/blb_sam_masks",          # _mask.png files
    "output_dir":      "/content/drive/MyDrive/blb_lesion_patches",     # output patches
    "img_exts":        (".jpg", ".jpeg", ".png"),

    # Tiny lesion handling
    # If the lesion bounding box is smaller than this (px), we zoom it up
    # so it's actually visible and usable. Set to 0 to disable.
    "min_patch_size":  64,    # minimum width OR height of output patch (px)
    "zoom_tiny":       True,  # zoom tiny patches up to min_patch_size

    # Preview run — set to True to process only first N images
    "preview_mode":    True,
    "preview_n":       10,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Config ready ✅")
print(f"Preview mode : {'ON — first ' + str(CONFIG['preview_n']) + ' images' if CONFIG['preview_mode'] else 'OFF — full dataset'}")

In [ ]:
# ============================================================
# Cell 3 — Build image-mask pairs

In [ ]:
# ============================================================

def build_pairs(image_dir, mask_dir, img_exts):
    """
    Match every _mask.png → original image by stem name.
    BLB_0418_mask.png  →  BLB_0418.jpg
    """
    pairs        = []
    missing_imgs = []

    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith("_mask.png")])

    for mf in mask_files:
        stem     = mf.replace("_mask.png", "")
        img_path = None
        for ext in img_exts:
            candidate = os.path.join(image_dir, stem + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path:
            pairs.append({"stem": stem, "img_path": img_path,
                           "mask_path": os.path.join(mask_dir, mf)})
        else:
            missing_imgs.append(stem)

    print(f"Pairs found   : {len(pairs)}")
    if missing_imgs:
        print(f"Missing images: {len(missing_imgs)} → {missing_imgs[:5]}")
    return pairs

all_pairs = build_pairs(CONFIG["image_dir"], CONFIG["mask_dir"], CONFIG["img_exts"])

# Slice for preview mode
pairs = all_pairs[:CONFIG["preview_n"]] if CONFIG["preview_mode"] else all_pairs
print(f"Processing    : {len(pairs)} images")

In [ ]:
# ============================================================
# Cell 4 — Core extraction function

In [ ]:
# ============================================================

def extract_lesion_patch(img_path, mask_path, stem, output_dir, cfg):
    """
    Extracts lesion region as a tight-cropped RGBA PNG (transparent background).

    Handles:
    - Mask/image size mismatch (resize mask to original resolution)
    - Tiny lesions (zoom up to cfg['min_patch_size'] if cfg['zoom_tiny'])
    - Multi-lesion masks (bounding rect covers all contours)
    - Morphological closing to fill small holes in mask
    - 5px padding around crop so edges aren't clipped

    Returns: dict with metadata (stem, patch_size, lesion_area_px, is_tiny, status)
    """
    result = {"stem": stem, "status": "ok", "is_tiny": False,
              "patch_w": 0, "patch_h": 0, "lesion_area_px": 0}

    # ── Load ────────────────────────────────────────────────
    img_bgr = cv2.imread(img_path)
    mask    = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img_bgr is None:
        result["status"] = "error_img"
        return result
    if mask is None:
        result["status"] = "error_mask"
        return result

    img_h, img_w = img_bgr.shape[:2]

    # ── Resize mask to original image size if needed ────────
    if mask.shape[:2] != (img_h, img_w):
        mask = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)

    # ── Clean binary threshold ───────────────────────────────
    _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # ── Morphological closing — fill small holes ─────────────
    # Use a smaller kernel (3×3) so tiny lesions aren't bloated
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel)

    # ── Check lesion area ────────────────────────────────────
    lesion_px = int(np.sum(mask_bin > 0))
    result["lesion_area_px"] = lesion_px

    if lesion_px == 0:
        result["status"] = "empty_mask"
        return result

    # ── Find contours → unified bounding rect ────────────────
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        result["status"] = "no_contour"
        return result

    x_min, y_min = img_w, img_h
    x_max, y_max = 0, 0
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        x_min = min(x_min, x)
        y_min = min(y_min, y)
        x_max = max(x_max, x + w)
        y_max = max(y_max, y + h)

    # 5px padding (don't clip edges)
    pad   = 5
    x_min = max(0, x_min - pad)
    y_min = max(0, y_min - pad)
    x_max = min(img_w, x_max + pad)
    y_max = min(img_h, y_max + pad)

    box_w = x_max - x_min
    box_h = y_max - y_min

    # ── Tiny lesion check ─────────────────────────────────────
    min_size = cfg.get("min_patch_size", 64)
    if box_w < min_size or box_h < min_size:
        result["is_tiny"] = True

    # ── Build RGBA ────────────────────────────────────────────
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    alpha   = mask_bin
    rgba    = np.dstack([img_rgb, alpha]).astype(np.uint8)

    # ── Crop ──────────────────────────────────────────────────
    patch = rgba[y_min:y_max, x_min:x_max]

    # ── Zoom tiny patches ─────────────────────────────────────
    if result["is_tiny"] and cfg.get("zoom_tiny", True):
        scale     = max(min_size / box_w, min_size / box_h)
        new_w     = max(int(box_w * scale), min_size)
        new_h     = max(int(box_h * scale), min_size)
        # Resize RGB and alpha separately to avoid alpha bleeding
        rgb_up    = cv2.resize(patch[:, :, :3], (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        alpha_up  = cv2.resize(patch[:, :, 3],  (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        patch     = np.dstack([rgb_up, alpha_up]).astype(np.uint8)

    result["patch_w"] = patch.shape[1]
    result["patch_h"] = patch.shape[0]

    # ── Save ──────────────────────────────────────────────────
    out_path = os.path.join(output_dir, stem + "_lesion.png")
    Image.fromarray(patch, mode="RGBA").save(out_path, format="PNG")

    return result

In [ ]:
# ============================================================
# Cell 5 — Run extraction

In [ ]:
# ============================================================

results  = []
success  = 0
failed   = 0
tiny     = 0
skipped  = 0

for i, p in enumerate(pairs):
    stem     = p["stem"]
    out_path = os.path.join(CONFIG["output_dir"], stem + "_lesion.png")

    if os.path.exists(out_path) and not CONFIG["preview_mode"]:
        skipped += 1
        continue

    r = extract_lesion_patch(
        p["img_path"], p["mask_path"], stem,
        CONFIG["output_dir"], CONFIG
    )
    results.append(r)

    if r["status"] == "ok":
        success += 1
        if r["is_tiny"]:
            tiny += 1
        flag = " ⚠️ tiny" if r["is_tiny"] else ""
        print(f"  [{i+1:>3}] {stem:<30} | {r['patch_w']}×{r['patch_h']}px | "
              f"lesion={r['lesion_area_px']:>6}px²{flag}")
    else:
        failed += 1
        print(f"  [{i+1:>3}] {stem:<30} | ❌ {r['status']}")

print(f"\n{'='*55}")
print(f"✅ Success : {success}  |  ⚠️  Tiny : {tiny}  |  ❌ Failed : {failed}  |  ⏭ Skipped : {skipped}")

In [ ]:
# ============================================================
# Cell 6 — Visual preview of all processed patches

In [ ]:
# ============================================================

patch_files = [r["stem"] + "_lesion.png" for r in results if r["status"] == "ok"]

if not patch_files:
    print("No patches to preview.")
else:
    n      = len(patch_files)
    ncols  = min(5, n)
    nrows  = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, fname in zip(axes, patch_files):
        patch     = np.array(Image.open(os.path.join(CONFIG["output_dir"], fname)))
        # Composite on white background for display
        white     = np.ones_like(patch[:, :, :3]) * 255
        a         = patch[:, :, 3:4] / 255.0
        composite = (patch[:, :, :3] * a + white * (1 - a)).astype(np.uint8)

        stem = fname.replace("_lesion.png", "")
        r    = next((x for x in results if x["stem"] == stem), {})
        tiny_flag = " ⚠️tiny" if r.get("is_tiny") else ""
        ax.imshow(composite)
        ax.set_title(f"{stem}\n{r.get('patch_w','')}×{r.get('patch_h','')}px{tiny_flag}",
                     fontsize=7)
        ax.axis("off")

    # Hide unused axes
    for ax in axes[len(patch_files):]:
        ax.axis("off")

    plt.suptitle("Lesion Patches Preview (white background for display only)", fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Cell 7 — Switch to FULL dataset run

In [ ]:
# ============================================================
# After previewing and you're satisfied:
#   1. Set CONFIG["preview_mode"] = False  in Cell 2
#   2. Re-run Cell 2 → Cell 5
#   Already-saved patches are skipped automatically.
print("To run full dataset: set preview_mode = False in Cell 2 and re-run from Cell 2.")